# Time travel × external mappers — analysis (human; manuscript-ready)

This notebook consumes the cache bundles produced by:

- `00_build_time_travel_vs_external_mappers_cache.ipynb`

and exports figures/tables that are designed to *market IDTrack* in a reviewer-proof way:

1) **Why time travel is necessary** (naive external mapping degrades on historical IDs).
2) **What you gain after time travel** (external tools recover coverage but show backend variability).
3) **What IDTrack adds beyond coverage**: explicit ambiguity (`1→n`) and a stable snapshot boundary.

## Outputs

- Figures → `idtrack-manuscript/figures/` and mirrored under `idtrack/reproducibility/experiments/_outputs/`
- Tables  → `idtrack-manuscript/tables/` and mirrored under `idtrack/reproducibility/experiments/_outputs/`
- Delta heatmaps: `idtrack-manuscript/figures/fig_time_travel_vs_external_mappers_deltas.pdf`
- Delta table: `idtrack-manuscript/tables/time_travel_vs_external_mappers_deltas.csv`

If you want an even deeper version, increase the grid density and bootstraps in the stage-0 cache notebook;
this analysis notebook automatically aggregates over bootstraps and stays the same.


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    atomic_write_dataframe_csv,
    notebook_context,
    read_json,
    read_pickle,
    save_figure,
    external_method_palette,
    EXTERNAL_MAPPER_METHODS_ORDERED,
)

from idtrack_results import (  # noqa: E402
    external_df_to_output_sets,
    jaccard_similarity,
    matchings_to_output_sets,
    summarize_matchings,
)

ctx = notebook_context('time_travel_vs_external_mappers', start=REPO_ROOT)

CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures
MANUSCRIPT_TABLES = ctx.manuscript_tables
EXPERIMENT_TABLES = (ctx.experiment_outputs / 'tables')

print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Load the latest cache fingerprint --------------------

params_candidates = list(CACHE_DIR.glob('time_travel_vs_external_mappers_params_*.json'))
if not params_candidates:
    raise FileNotFoundError(f'No params JSON found under {CACHE_DIR}. Run the stage-0 cache notebook.')

PARAMS_JSON = max(params_candidates, key=lambda p: p.stat().st_mtime)
fp = PARAMS_JSON.stem.split('_')[-1]
POOLS_JSON = CACHE_DIR / f'time_travel_vs_external_mappers_pools_{fp}.json'

params = read_json(PARAMS_JSON)
print('Using:', PARAMS_JSON)
print('Fingerprint:', fp)
print('Pools:', POOLS_JSON if POOLS_JSON.exists() else '(missing)')
pd.DataFrame([params])


In [ ]:
# -------------------- Load bundles --------------------

bundles = []
for p in sorted(CACHE_DIR.glob(f'time_travel_vs_external_mappers_bundle_{fp}_from*_b*.pickle')):
    try:
        bundles.append(read_pickle(p))
    except Exception as e:  # noqa: BLE001
        print('WARN: failed to read', p.name, ':', e)

print('Loaded bundles:', len(bundles))
if not bundles:
    raise FileNotFoundError('No bundle caches found. Run the stage-0 cache notebook.')

# Basic integrity check
print('Targets:', params.get('target_databases'))
print('External methods:', params.get('external_methods'))


In [ ]:
# -------------------- Summarize outcomes + agreement --------------------

TARGETS = list(params.get('target_databases') or [])
EXTERNAL_METHODS = [str(m).strip().lower() for m in (params.get('external_methods') or [])]

def _external_outcome_summary(df: pd.DataFrame, inputs: list[str]) -> dict[str, float | int]:
    inputs = [str(x) for x in (inputs or [])]
    if not inputs:
        return {'n_total': 0, 'n_1_to_0': 0, 'n_1_to_1': 0, 'n_1_to_n': 0, 'frac_1_to_0': float('nan'), 'frac_1_to_1': float('nan'), 'frac_1_to_n': float('nan')}
    if df is None or df.empty:
        n = len(set(inputs))
        return {'n_total': n, 'n_1_to_0': n, 'n_1_to_1': 0, 'n_1_to_n': 0, 'frac_1_to_0': 1.0, 'frac_1_to_1': 0.0, 'frac_1_to_n': 0.0}
    per = df.drop_duplicates('input_id')
    vc = per['mapping'].value_counts().to_dict()
    n_total = len(set(inputs))
    n_10 = int(vc.get('1:0', 0))
    n_11 = int(vc.get('1:1', 0))
    n_1n = int(vc.get('1:n', 0))
    den = float(n_total) if n_total else float('nan')
    return {
        'n_total': n_total,
        'n_1_to_0': n_10,
        'n_1_to_1': n_11,
        'n_1_to_n': n_1n,
        'frac_1_to_0': n_10 / den if n_total else float('nan'),
        'frac_1_to_1': n_11 / den if n_total else float('nan'),
        'frac_1_to_n': n_1n / den if n_total else float('nan'),
    }

def _mean_jaccard(ref_sets: dict[str, set[str]], other_sets: dict[str, set[str]], inputs: list[str]) -> float:
    if not inputs:
        return float('nan')
    vals = []
    for q in inputs:
        a = ref_sets.get(str(q), set())
        b = other_sets.get(str(q), set())
        vals.append(jaccard_similarity(a, b))
    return float(np.nanmean(vals)) if vals else float('nan')

rows = []
for bundle in bundles:
    fr = int(bundle.get('from_release'))
    b = int(bundle.get('bootstrap'))
    ids_from = [str(x) for x in (bundle.get('ids_from') or [])]
    to_ids = [str(x) for x in (bundle.get('to_ids_1to1_unique') or [])]

    # Backbone recoverability
    n_backbone_1to1 = int(bundle.get('n_backbone_1to1') or 0)
    rows.append({
        'from_release': fr,
        'bootstrap': b,
        'target_db': '(backbone)',
        'scenario': 'backbone',
        'method': 'IDTrack',
        'n_inputs': len(ids_from),
        'n_backbone_1to1': n_backbone_1to1,
        'n_to_ids_unique': len(to_ids),
        'frac_backbone_1to1': (n_backbone_1to1 / len(ids_from)) if ids_from else float('nan'),
    })

    idt_old = bundle.get('idtrack_old_to_target_matchings') or {}
    idt_to = bundle.get('idtrack_to_to_target_matchings') or {}
    ext = bundle.get('external_results') or {}

    for target in TARGETS:
        # IDTrack references
        m_old = idt_old.get(target) or []
        m_to = idt_to.get(target) or []

        s_old = summarize_matchings(m_old)
        s_to = summarize_matchings(m_to)

        rows.append({
            'from_release': fr,
            'bootstrap': b,
            'target_db': target,
            'scenario': 'idtrack_old_to_target',
            'method': 'IDTrack',
            'n_inputs': len(ids_from),
            **{k: v for k, v in s_old.items() if k.startswith('frac_')},
        })

        rows.append({
            'from_release': fr,
            'bootstrap': b,
            'target_db': target,
            'scenario': 'idtrack_to_to_target',
            'method': 'IDTrack',
            'n_inputs': len(to_ids),
            **{k: v for k, v in s_to.items() if k.startswith('frac_')},
        })

        # Agreement references
        ref_old_sets = matchings_to_output_sets(m_old)
        ref_to_sets = matchings_to_output_sets(m_to)

        # External: naive
        ext_naive = (ext.get('naive') or {}).get(target) or {}
        for method, df in ext_naive.items():
            method = str(method).lower()
            summ = _external_outcome_summary(df, ids_from)
            other_sets = external_df_to_output_sets(df, inputs=ids_from)
            rows.append({
                'from_release': fr,
                'bootstrap': b,
                'target_db': target,
                'scenario': 'external_naive',
                'method': method,
                'n_inputs': len(ids_from),
                **{k: v for k, v in summ.items() if k.startswith('frac_')},
                'mean_jaccard_vs_idtrack': _mean_jaccard(ref_old_sets, other_sets, ids_from),
            })

        # External: time-travel assisted
        ext_tt = (ext.get('time_travel_assisted') or {}).get(target) or {}
        for method, df in ext_tt.items():
            method = str(method).lower()
            summ = _external_outcome_summary(df, to_ids)
            other_sets = external_df_to_output_sets(df, inputs=to_ids)
            rows.append({
                'from_release': fr,
                'bootstrap': b,
                'target_db': target,
                'scenario': 'external_time_travel_assisted',
                'method': method,
                'n_inputs': len(to_ids),
                **{k: v for k, v in summ.items() if k.startswith('frac_')},
                'mean_jaccard_vs_idtrack': _mean_jaccard(ref_to_sets, other_sets, to_ids),
            })

summary = pd.DataFrame(rows)
summary = summary.sort_values(['scenario', 'target_db', 'method', 'from_release', 'bootstrap']).reset_index(drop=True)
summary.head(20)


In [ ]:
# -------------------- Aggregate over bootstraps --------------------

numeric_cols = [c for c in summary.columns if c.startswith('frac_') or c.startswith('mean_') or c.startswith('n_')]
group_cols = ['scenario', 'target_db', 'method', 'from_release']

mean = summary.groupby(group_cols, as_index=False)[numeric_cols].mean(numeric_only=True)
std = summary.groupby(group_cols, as_index=False)[numeric_cols].std(numeric_only=True).rename(columns={c: f'std_{c}' for c in numeric_cols})
agg = mean.merge(std, on=group_cols, how='left')

agg.head(20)


In [ ]:
# -------------------- Export summary tables --------------------

out_csv = MANUSCRIPT_TABLES / 'time_travel_vs_external_mappers_summary.csv'
atomic_write_dataframe_csv(agg, out_csv, index=False)
atomic_write_dataframe_csv(agg, EXPERIMENT_TABLES / out_csv.name, index=False)
print('Wrote:', out_csv)


In [ ]:
# -------------------- Figure: backbone recoverability vs time --------------------

back = agg[(agg['scenario'] == 'backbone') & (agg['method'] == 'IDTrack')].copy()
if back.empty:
    print('No backbone rows.')
else:
    fig, ax = plt.subplots(1, 1, figsize=(8.8, 3.8), constrained_layout=True)
    ax.plot(back['from_release'], back['frac_backbone_1to1'], '-o', lw=1.6, ms=3, color=MANUSCRIPT_COLORS['IDTrack'])
    ax.set_ylim(0, 1)
    ax.set_xlabel('from_release (historical Ensembl)')
    ax.set_ylabel('Fraction with 1→1 backbone time travel')
    ax.set_title('Backbone recoverability into the target time boundary')
    written = save_figure(fig, 'fig_time_travel_backbone_recoverability.pdf', ctx, formats=('pdf',))
    print('Saved:', written['pdf'])


In [ ]:
# -------------------- Figure: external coverage + agreement (multi-panel) --------------------

def _plot_panel(ax, sub: pd.DataFrame, *, y: str, title: str):
    # Use stable method ordering when present
    methods = [m for m in EXTERNAL_MAPPER_METHODS_ORDERED if m.lower() in set(sub['method'])] + [m for m in sorted(set(sub['method'])) if m.lower() not in {x.lower() for x in EXTERNAL_MAPPER_METHODS_ORDERED}]
    colors = external_method_palette([m if m in MANUSCRIPT_COLORS else m for m in methods])

    for m, c in zip(methods, colors, strict=False):
        key = m.lower()
        d = sub[sub['method'] == key]
        if d.empty:
            continue
        ax.errorbar(
            d['from_release'],
            d[y],
            yerr=d.get(f'std_{y}', None),
            fmt='-o',
            lw=1.2,
            ms=3,
            label=m,
            color=c,
        )
    ax.set_ylim(0, 1)
    ax.set_xlabel('from_release')
    ax.set_title(title)

for target in TARGETS:
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2), constrained_layout=True)
    axes = np.ravel(axes)

    # Naive external mapping: coverage loss
    sub_naive = agg[(agg['scenario'] == 'external_naive') & (agg['target_db'] == target)].copy()
    _plot_panel(axes[0], sub_naive, y='frac_1_to_0', title=f'Naive external mapping — 1→0 fraction ({target})')
    axes[0].set_ylabel('Fraction unmapped (1→0)')

    _plot_panel(axes[1], sub_naive, y='mean_jaccard_vs_idtrack', title=f'Naive external mapping — agreement vs IDTrack ({target})')
    axes[1].set_ylabel('Mean Jaccard vs IDTrack')

    # Time-travel assisted
    sub_tt = agg[(agg['scenario'] == 'external_time_travel_assisted') & (agg['target_db'] == target)].copy()
    _plot_panel(axes[2], sub_tt, y='frac_1_to_0', title=f'Time-travel assisted — 1→0 fraction ({target})')
    axes[2].set_ylabel('Fraction unmapped (1→0)')

    _plot_panel(axes[3], sub_tt, y='mean_jaccard_vs_idtrack', title=f'Time-travel assisted — agreement vs IDTrack ({target})')
    axes[3].set_ylabel('Mean Jaccard vs IDTrack')

    for ax in axes:
        ax.grid(True, color=MANUSCRIPT_COLORS['grid'], lw=0.5)
        ax.legend(frameon=True, fontsize=8)

    fname = 'fig_time_travel_vs_external_mappers_' + ('hgnc' if 'HGNC' in target else 'uniprot') + '.pdf'
    written = save_figure(fig, fname, ctx, formats=('pdf',))
    print('Saved:', written['pdf'])


In [ ]:
# -------------------- Figure: coverage heatmaps (method × from_release) --------------------

def _coverage_heatmap(df: pd.DataFrame, *, title: str):
    if df.empty:
        return None
    pivot = df.pivot(index='method', columns='from_release', values='frac_1_to_0')
    pivot = pivot.loc[[m for m in pivot.index if m in [x.lower() for x in EXTERNAL_MAPPER_METHODS_ORDERED]] + [m for m in pivot.index if m not in [x.lower() for x in EXTERNAL_MAPPER_METHODS_ORDERED]]]
    fig, ax = plt.subplots(1, 1, figsize=(12.2, 2.8 + 0.35 * len(pivot.index)), constrained_layout=True)
    if sns is not None:
        sns.heatmap(pivot, ax=ax, cmap='Reds', vmin=0, vmax=1, linewidths=0.25, linecolor=MANUSCRIPT_COLORS['grid'])
    else:
        im = ax.imshow(pivot.values, vmin=0, vmax=1, cmap='Reds', aspect='auto')
        ax.figure.colorbar(im, ax=ax, shrink=0.7)
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_xticklabels([str(c) for c in pivot.columns], rotation=45, ha='right')
        ax.set_yticklabels(list(pivot.index))
    ax.set_title(title)
    ax.set_xlabel('from_release')
    ax.set_ylabel('method')
    return fig

for target in TARGETS:
    for scenario, label in [('external_naive', 'naive'), ('external_time_travel_assisted', 'time_travel')]:
        sub = agg[(agg['scenario'] == scenario) & (agg['target_db'] == target)].copy()
        fig = _coverage_heatmap(sub, title=f'External coverage failure (1→0) — {label} — {target}')
        if fig is None:
            continue
        fname = f'fig_time_travel_vs_external_mappers_heatmap_{label}_' + ('hgnc' if 'HGNC' in target else 'uniprot') + '.pdf'
        written = save_figure(fig, fname, ctx, formats=('pdf',))
        print('Saved:', written['pdf'])


## Marketing extension: quantify the *gain* from time travel (delta heatmaps)

The main multi-panel figures show the raw trends. For a Results section, it is often even more convincing to show the **difference** between:

- **Naive external mapping** (old Ensembl IDs at historical releases)
- **Time-travel assisted mapping** (old → `to_release` with IDTrack, then external mapping)

This produces a clean, reviewer-proof statement:

> Time travel is not a cosmetic feature; it is a measurable intervention that restores coverage and (often) improves agreement.


In [ ]:
from external_mappers_analysis import scenario_delta_table  # noqa: E402
from experiments_utils import label_panels  # noqa: E402
from plotting_utils import heatmap  # noqa: E402

# Build per-(target, method, from_release) deltas: (time_travel_assisted - naive)
deltas = scenario_delta_table(
    agg,
    scenario_a='external_naive',
    scenario_b='external_time_travel_assisted',
    keys=['target_db', 'method', 'from_release'],
    metrics=['frac_1_to_0', 'mean_jaccard_vs_idtrack'],
)

if deltas.empty:
    print('No delta rows; ensure both scenarios are present in agg.')
else:
    # Human-friendly signed gains
    deltas['gain_coverage'] = -deltas['delta_frac_1_to_0']  # positive = fewer 1→0 after time travel
    deltas['gain_jaccard'] = deltas['delta_mean_jaccard_vs_idtrack']  # positive = higher agreement

    out_delta = MANUSCRIPT_TABLES / 'time_travel_vs_external_mappers_deltas.csv'
    atomic_write_dataframe_csv(deltas, out_delta, index=False)
    atomic_write_dataframe_csv(deltas, EXPERIMENT_TABLES / out_delta.name, index=False)
    print('Wrote:', out_delta)

    targets = TARGETS[:2] if TARGETS else sorted(deltas['target_db'].unique().tolist())
    ncols = max(1, len(targets))

    fig, axes = plt.subplots(2, ncols, figsize=(6.5 * ncols, 7.0), constrained_layout=True)
    if ncols == 1:
        axes = np.array(axes).reshape(2, 1)

    for j, target in enumerate(targets):
        sub = deltas[deltas['target_db'] == target].copy()
        if sub.empty:
            axes[0, j].axis('off')
            axes[1, j].axis('off')
            axes[0, j].text(0.5, 0.5, f'No delta data for {target}', ha='center', va='center')
            continue

        # stable method ordering
        ordered = [m.lower() for m in EXTERNAL_MAPPER_METHODS_ORDERED if m.lower() in set(sub['method'])]
        ordered += [m for m in sorted(set(sub['method'])) if m not in set(ordered)]

        mat_cov = sub.pivot(index='method', columns='from_release', values='gain_coverage').reindex(ordered)
        mat_jac = sub.pivot(index='method', columns='from_release', values='gain_jaccard').reindex(ordered)

        cov_absmax = float(np.nanmax(np.abs(mat_cov.values))) if mat_cov.size else 0.0
        jac_absmax = float(np.nanmax(np.abs(mat_jac.values))) if mat_jac.size else 0.0
        cov_absmax = max(cov_absmax, 0.05)
        jac_absmax = max(jac_absmax, 0.05)

        heatmap(
            axes[0, j],
            mat_cov,
            title=f'Coverage gain after time travel (−Δ 1→0)\n{target}',
            cmap='RdBu_r',
            vmin=-cov_absmax,
            vmax=cov_absmax,
            center=0.0,
            cbar=True,
            cbar_label='gain',
            square=False,
        )
        axes[0, j].set_xlabel('from_release')
        axes[0, j].set_ylabel('method')

        heatmap(
            axes[1, j],
            mat_jac,
            title=f'Agreement gain after time travel (Δ Jaccard vs IDTrack)\n{target}',
            cmap='RdBu_r',
            vmin=-jac_absmax,
            vmax=jac_absmax,
            center=0.0,
            cbar=True,
            cbar_label='gain',
            square=False,
        )
        axes[1, j].set_xlabel('from_release')
        axes[1, j].set_ylabel('method')

    label_panels(axes.ravel())
    written = save_figure(fig, 'fig_time_travel_vs_external_mappers_deltas.pdf', ctx, formats=('pdf',))
    print('Saved:', written['pdf'])
